In [1]:
silver_df = spark.sql("""
    SELECT *
    FROM silver.complaints
""")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 3, Finished, Available, Finished, False)

In [2]:
display(silver_df.limit(10))

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a97041a4-6824-40a4-aa65-3d1816c7d3a9)

In [3]:
silver_df.printSchema()

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 5, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- date_received_clean: date (nullable = true)
 |-- date_sent_to_company_clean: date (nullable = true)
 |-- state_clean: string (nullabl

## Create Gold Schema

In [4]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 6, Finished, Available, Finished, False)

DataFrame[]

## Create Date Dimension

In [5]:
dim_date_df = spark.sql("""
    SELECT
        date,
        YEAR(date) AS year,
        QUARTER(date) AS quarter,
        MONTH(date) AS month_number,
        DATE_FORMAT(date, 'MMMM') AS month_name,
        DATE_FORMAT(date, 'yyyy-MM') AS year_month,
        DAYOFWEEK(date) AS day_of_week_number,
        DATE_FORMAT(date, 'EEEE') AS day_of_week_name,
        CASE
            WHEN DAYOFWEEK(date) IN (1, 7) THEN true
            ELSE false
        END AS is_weekend
    FROM (
        SELECT
            EXPLODE(
                SEQUENCE(
                    MIN(date_received_clean),
                    MAX(date_received_clean),
                    INTERVAL 1 DAY
                )
            ) AS date
        FROM silver.complaints
    ) date_base
    ORDER BY date
""")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 7, Finished, Available, Finished, False)

In [6]:
display(dim_date_df.limit(10))

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 141b73e6-5c5f-4253-9765-fe477e5e5ec6)

In [7]:
print(f"Date dimension rows: {dim_date_df.count():,}")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 9, Finished, Available, Finished, False)

Date dimension rows: 5,324


In [8]:
(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date")
)

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 10, Finished, Available, Finished, False)

- I created the date dimension from the minimum and maximum received dates.
- The table includes calendar fields such as year, quarter, month, day of week, and weekend flag for trend analysis.

## Create Location Dimension

In [9]:
dim_location_df = spark.sql("""
    SELECT DISTINCT
        CONCAT(
            COALESCE(state_clean, 'Unknown'),
            '|',
            COALESCE(zip_code_clean, 'No ZIP'),
            '|',
            zip_code_status
        ) AS location_key,

        COALESCE(state_clean, 'Unknown') AS state,
        COALESCE(zip_code_clean, 'No ZIP') AS zip_code,
        zip_code_status
    FROM silver.complaints
    ORDER BY state, zip_code, zip_code_status
""")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 11, Finished, Available, Finished, False)

In [12]:
display(dim_location_df.limit(10))

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c870fcc2-27ba-4ad8-b852-2b116645e403)

In [11]:
print(f"Location dimension rows: {dim_location_df.count():,}")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 13, Finished, Available, Finished, False)

Location dimension rows: 40,491


In [13]:
(
    dim_location_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_location")
)

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 15, Finished, Available, Finished, False)

In [14]:
spark.sql("DESCRIBE TABLE gold.dim_location").show(truncate=False)

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 16, Finished, Available, Finished, False)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|location_key   |string   |NULL   |
|state          |string   |NULL   |
|zip_code       |string   |NULL   |
|zip_code_status|string   |NULL   |
+---------------+---------+-------+



- This table uses location_key to uniquely identify each combination of state, ZIP code, and ZIP code status.
- An official ZIP code reference dataset needs to be used to validate whether ZIP values actually exist and to enrich locations with city, region, latitude, longitude, and population data.

## Create Product and Issue Dimension

In [15]:
dim_product_issue_df = spark.sql("""
    SELECT DISTINCT
        CONCAT(
            COALESCE(product, 'Unknown'),
            '|',
            COALESCE(sub_product, 'No Sub-product'),
            '|',
            COALESCE(issue, 'Unknown'),
            '|',
            COALESCE(sub_issue, 'No Sub-issue')
        ) AS product_issue_key,

        COALESCE(product, 'Unknown') AS product,
        COALESCE(sub_product, 'No Sub-product') AS sub_product,
        COALESCE(issue, 'Unknown') AS issue,
        COALESCE(sub_issue, 'No Sub-issue') AS sub_issue

    FROM silver.complaints
    ORDER BY product, sub_product, issue, sub_issue
""")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 17, Finished, Available, Finished, False)

In [18]:
display(dim_product_issue_df.limit(20))

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f47d6be0-1c33-4fd9-9430-685164e9c336)

In [19]:
print(f"Product/issue dimension rows: {dim_product_issue_df.count():,}")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 21, Finished, Available, Finished, False)

Product/issue dimension rows: 2,643


In [20]:
(
    dim_product_issue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_product_issue")
)

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 22, Finished, Available, Finished, False)

## Create Complaint Fact Table

In [23]:
fact_complaints_df = spark.sql("""
    SELECT
        complaint_id,

        date_received_clean AS date_received,
        date_sent_to_company_clean AS date_sent_to_company,

        CONCAT(
            COALESCE(state_clean, 'Unknown'),
            '|',
            COALESCE(zip_code_clean, 'No ZIP'),
            '|',
            zip_code_status
        ) AS location_key,

        CONCAT(
            COALESCE(product, 'Unknown'),
            '|',
            COALESCE(sub_product, 'No Sub-product'),
            '|',
            COALESCE(issue, 'Unknown'),
            '|',
            COALESCE(sub_issue, 'No Sub-issue')
        ) AS product_issue_key,

        company,
        submitted_via,
        company_response_to_consumer,
        timely_response,

        DATEDIFF(date_sent_to_company_clean, date_received_clean) AS response_days

    FROM silver.complaints
""")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 25, Finished, Available, Finished, False)

In [24]:
display(fact_complaints_df.limit(50))

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b440372b-5124-48c0-93b4-88abed651b19)

In [25]:
print(f"Complaint fact rows: {fact_complaints_df.count():,}")

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 27, Finished, Available, Finished, False)

Complaint fact rows: 16,279,252


In [28]:
(
    fact_complaints_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_complaints")
)

StatementMeta(, e06d0c79-7883-410c-b766-077ed13833a6, 30, Finished, Available, Finished, False)